<a href="https://colab.research.google.com/github/sttr47/customer-feedback-classification/blob/main/apar_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.disabled = False

set_seed(42)

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/train.csv')
df

,id,feedback,tag,label
0,0,bugun 2 velosipod goturdum ikisinin de tormozu...,NaN,technical_support
1,1,sol pedalda problem var bu veloda,NaN,technical_support
2,2,Oturacağın alt-ön tərəfi çıxıb dəyişilməlidir,Other,technical_support
3,3,velospied kilidə düşmürdü,NaN,technical_support
4,4,Gprs islemir anbara getmelidi,Electrical system,technical_support
...,...,...,...,...
17764,17764,one pedal was broken,NaN,technical_support
17765,17765,Больше 7 км в час не разгоняется,Gears/transmission,technical_support
17766,17766,yağlanma lazımdı,Other,technical_support
17767,17767,он не включается уже какой по счету,NaN,technical_support


In [ ]:
df['tag'].isnull().sum()

np.int64(12504)

In [ ]:
df['label'].value_counts()

,count
label,
technical_support,10642
other,3800
customer_support,3327


In [ ]:
df['tag'] = df['tag'].fillna('')

In [ ]:
df['feedback'] = df['feedback'].fillna('')

In [ ]:
text = df.apply(lambda row: f"{row['tag']}, {row['feedback']}" if row['tag'] else row['feedback'], axis=1)
text

,0
0,bugun 2 velosipod goturdum ikisinin de tormozu...
1,sol pedalda problem var bu veloda
2,"Other, Oturacağın alt-ön tərəfi çıxıb dəyişi..."
3,velospied kilidə düşmürdü
4,"Electrical system, Gprs islemir anbara getmelidi"
...,...
17764,one pedal was broken
17765,"Gears/transmission, Больше 7 км в час не разго..."
17766,"Other, yağlanma lazımdı"
17767,он не включается уже какой по счету


# 1) ML Baseline

In [ ]:
X = df[['tag', 'feedback']]
y = df['label']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer([
    ('tag_ohe', OneHotEncoder(handle_unknown='ignore'), ['tag']),
    ('feedback_tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000), 'feedback')
])

In [ ]:
model = LogisticRegression(C=2.0, class_weight='balanced', random_state=42)

full_pipeline = Pipeline([
    ('preprocessing', transformer),
    ('estimator', model)
])

In [ ]:
# full_pipeline.fit(df[['tag', 'feedback']], df['label'])

In [ ]:
# print(f'train_score: {full_pipeline.score(X_train, y_train)*100}, test_score: {full_pipeline.score(X_test, y_test)*100}')

# 2) Transformer

In [ ]:
from torch.utils.data import Dataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

In [ ]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
label_dict = {'customer_support': 0, 'other': 1, 'technical_support': 2}

df['label'] = df['label'].map(label_dict)

df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print("Unikal labels:", df['label'].unique())

Unikal labels: [2 0 1]


In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
class DataRaceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# for appearence
train_texts = train_df.apply(lambda row: f"{row['tag']}, {row['feedback']}" if row['tag'] else row['feedback'], axis=1).values
val_texts = val_df.apply(lambda row: f"{row['tag']}, {row['feedback']}" if row['tag'] else row['feedback'], axis=1).values

train_dataset = DataRaceDataset(
    texts=train_texts,
    labels=train_df['label'].values,
    tokenizer=tokenizer
)

val_dataset = DataRaceDataset(
    texts=val_texts,
    labels=val_df['label'].values,
    tokenizer=tokenizer
)

In [ ]:
device = 'cpu'
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': macro_f1}

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.bert.encoder.layer[-1].parameters():
    param.requires_grad = True

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    fp16=True,
    seed=42,
    data_seed=42,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Training...")
trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training...


Epoch,Training Loss,Validation Loss,Macro F1
1,0.458581,0.432483,0.787524
2,0.436477,0.401983,0.807281
3,0.330288,0.379615,0.820306


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1335, training_loss=0.4201941615186827, metrics={'train_runtime': 82.1181, 'train_samples_per_second': 519.313, 'train_steps_per_second': 16.257, 'total_flos': 1402558962376320.0, 'train_loss': 0.4201941615186827, 'epoch': 3.0})

In [ ]:
raw_predictions = trainer.predict(val_dataset)

import torch.nn.functional as F
transformer_probs = F.softmax(torch.tensor(raw_predictions.predictions), dim=1).numpy()

print("Transformer ehtimallar matrisinin forması:", transformer_probs.shape)

Transformer ehtimallar matrisinin forması: (3554, 3)


In [ ]:
full_pipeline.fit(train_df[['tag', 'feedback']], train_df['label'])
print(f"train_score: {full_pipeline.score(train_df[['tag','feedback']], train_df['label'])*100}, "
      f"val_score: {full_pipeline.score(val_df[['tag','feedback']], val_df['label'])*100}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


train_score: 92.2335561027084, val_score: 85.28418683173888


In [ ]:
ml_probs = full_pipeline.predict_proba(val_df[['tag', 'feedback']])

print("ML prob shapes:", ml_probs.shape)

ML prob shapes: (3554, 3)


In [ ]:
print(full_pipeline.classes_)

[0 1 2]


In [ ]:
true_labels = val_df['label'].values
best_f1 = 0
best_w = 0

for w in np.linspace(0, 1, 11):
    hybrid_probs = (w * transformer_probs) + ((1 - w) * ml_probs)

    hybrid_preds = np.argmax(hybrid_probs, axis=1)
    current_f1 = f1_score(true_labels, hybrid_preds, average='macro')
    print(f"Weight (Transformer: {w:.1f}, ML: {1-w:.1f}) -> Validation Macro F1: {current_f1:.4f}")

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_w = w

print(f"Best Score: {best_f1:.4f}")
print(f"Best Transformer Weight: {best_w:.1f}")


Weight (Transformer: 0.0, ML: 1.0) -> Validation Macro F1: 0.8304
Weight (Transformer: 0.1, ML: 0.9) -> Validation Macro F1: 0.8425
Weight (Transformer: 0.2, ML: 0.8) -> Validation Macro F1: 0.8526
Weight (Transformer: 0.3, ML: 0.7) -> Validation Macro F1: 0.8569
Weight (Transformer: 0.4, ML: 0.6) -> Validation Macro F1: 0.8650
Weight (Transformer: 0.5, ML: 0.5) -> Validation Macro F1: 0.8604
Weight (Transformer: 0.6, ML: 0.4) -> Validation Macro F1: 0.8507
Weight (Transformer: 0.7, ML: 0.3) -> Validation Macro F1: 0.8405
Weight (Transformer: 0.8, ML: 0.2) -> Validation Macro F1: 0.8323
Weight (Transformer: 0.9, ML: 0.1) -> Validation Macro F1: 0.8276
Weight (Transformer: 1.0, ML: 0.0) -> Validation Macro F1: 0.8203
Best Score: 0.8650
Best Transformer Weight: 0.4


In [ ]:
test_df = pd.read_csv("/content/test.csv")
test_df['tag'] = test_df['tag'].fillna("")
test_df['feedback'] = test_df['feedback'].fillna("")
test_df['text'] = test_df.apply(
    lambda row: f"{row['tag']}, {row['feedback']}" if row['tag'] else row['feedback'], axis=1
)

full_pipeline.fit(df[['tag', 'feedback']], df['label'])

test_ml_probs = full_pipeline.predict_proba(test_df[['tag', 'feedback']])

test_dataset = DataRaceDataset(
    texts=test_df['text'].values,
    labels=[0] * len(test_df),
    tokenizer=tokenizer
)

test_raw_preds = trainer.predict(test_dataset)

import torch.nn.functional as F
test_transformer_probs = F.softmax(torch.tensor(test_raw_preds.predictions), dim=1).numpy()

final_test_probs = (0.4 * test_transformer_probs) + (0.6 * test_ml_probs)
final_test_preds = np.argmax(final_test_probs, axis=1)

reverse_label_dict = {0: 'customer_support', 1: 'other', 2: 'technical_support'}
final_test_labels = [reverse_label_dict[pred] for pred in final_test_preds]

submission = pd.DataFrame({
    'id': test_df['id'],
    'label': final_test_labels
})

submission.to_csv("finall.csv", index=False)
print("Succesfully created: 'finall.csv'")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Succesfully created: 'finall.csv'


In [ ]:
from google.colab import files
files.download('finall.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>